# Fine-tune EfficientNet-B0 on Coffee Leaf Dataset
This notebook demonstrates a minimal fine-tuning workflow using PyTorch.
Use this as an example — adjust hyperparameters and dataset paths for real training.

In [1]:
# Install dependencies if needed
# !pip install torch torchvision timm scikit-learn pandas matplotlib

import os
from pathlib import Path
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'numpy'

## Prepare dataset paths and augmentations

In [ ]:
root = Path('..').resolve() / 'model'
test_dir = root / 'test_dataset'
train_aug = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
val_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

: 

## Minimal Dataset class

In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, root_dir, split='diseases', transform=None):
        self.samples = []
        self.transform = transform
        base = Path(root_dir) / split
        for class_dir in base.iterdir():
            if not class_dir.is_dir():
                continue
            label = class_dir.name.strip().lower()
            for p in class_dir.glob('*.jpg'):
                self.samples.append((str(p), label))
        self.classes = sorted(list({s[1] for s in self.samples}))
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, label = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.class_to_idx[label]

## Create dataloaders and model

In [ ]:
ds = SimpleImageDataset(root / 'test_dataset', split='diseases', transform=train_aug)
if len(ds) < 10:
    print('Not enough examples for fine-tuning in this demo. Replace dataset with larger training set to run this section.')
else:
    train_size = int(0.8 * len(ds))
    val_size = len(ds) - train_size
    train_ds, val_ds = random_split(ds, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    num_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(num_features, len(ds.classes))
    model = model.to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # Simple training loop (demo)
    for epoch in range(3):
        model.train()
        running = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        print(f'Epoch {epoch+1} train loss: {running/len(train_loader.dataset):.4f}')

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                out = model(xb)
                preds = out.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        print(f'Epoch {epoch+1} val acc: {correct/total if total else 0:.4f}')

    # Save a checkpoint
    torch.save(model.state_dict(), str(root / 'models' / 'leaf_diseases' / 'finetuned_demo.pth'))
    print('Saved finetuned model checkpoint (demo).')

## Notes
- This notebook is a minimal example for educational purposes. For production training, use larger datasets, better hyperparameter schedules, mixed precision, and proper logging.
- Use `torch.cuda.amp` and gradient accumulation for constrained GPUs.